In [43]:
!pip install -r requirements.txt -q

In [44]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
import shap
import joblib

In [45]:
import custom_class_copy as cc
import feature_engineering_functions as func

In [46]:
model = joblib.load('modelo_XGB_V2.joblib')

In [47]:
custom_model = cc.CustomPrediction(model)

In [48]:
data = [
    {"aerolinea": "DL", "aeropuerto_origen": "JFK", "aeropuerto_destino": "MSP", "fecha_vuelo": "2026-11-30 09:12", 'distancia': 435},
    {"aerolinea": "DL", "aeropuerto_origen": "JFK", "aeropuerto_destino": "MSP", "fecha_vuelo": "2026-11-30 13:12"},
    {"aerolinea": "DL", "aeropuerto_origen": "JFK", "aeropuerto_destino": "MSP", "fecha_vuelo": "2026-11-30 19:12", 'distancia': 435},
    {"aerolinea": "DL", "aeropuerto_origen": "JFK", "aeropuerto_destino": "MSP", "fecha_vuelo": "2026-11-30 00:12"},

    {"aerolinea": "AA", "aeropuerto_origen": "ORD", "aeropuerto_destino": "CLT", "fecha_vuelo": "2026-12-24 09:12", 'distancia': 812},
    {"aerolinea": "AA", "aeropuerto_origen": "ORD", "aeropuerto_destino": "CLT", "fecha_vuelo": "2026-12-24 13:12"},
    {"aerolinea": "AA", "aeropuerto_origen": "ORD", "aeropuerto_destino": "CLT", "fecha_vuelo": "2026-12-24 19:12", 'distancia': 812},
    {"aerolinea": "AA", "aeropuerto_origen": "ORD", "aeropuerto_destino": "CLT", "fecha_vuelo": "2026-12-24 00:12"},

    {"aerolinea": "B6", "aeropuerto_origen": "FLL", "aeropuerto_destino": "BNA", "fecha_vuelo": "2026-01-15 09:12"},
    {"aerolinea": "B6", "aeropuerto_origen": "FLL", "aeropuerto_destino": "BNA", "fecha_vuelo": "2026-01-15 13:12", 'distancia': 783},
    {"aerolinea": "B6", "aeropuerto_origen": "FLL", "aeropuerto_destino": "BNA", "fecha_vuelo": "2026-01-15 19:12", 'distancia': 783},
    {"aerolinea": "B6", "aeropuerto_origen": "FLL", "aeropuerto_destino": "BNA", "fecha_vuelo": "2026-01-15 00:12"},

    {"aerolinea": "DL", "aeropuerto_origen": "BHM", "aeropuerto_destino": "ATL", "fecha_vuelo": "2026-02-22 05:12", 'distancia': 1045},
    {"aerolinea": "DL", "aeropuerto_origen": "BHM", "aeropuerto_destino": "ATL", "fecha_vuelo": "2026-02-22 3:12"},
    {"aerolinea": "DL", "aeropuerto_origen": "BHM", "aeropuerto_destino": "ATL", "fecha_vuelo": "2026-02-22 20:12"},
    {"aerolinea": "DL", "aeropuerto_origen": "BHM", "aeropuerto_destino": "ATL", "fecha_vuelo": "2026-02-22 00:12", 'distancia': 1045}
]

In [49]:
# data = {"aerolinea": "DL", "aeropuerto_origen": "BHM", "aeropuerto_destino": "ATL", "fecha_vuelo": "2026-11-30 09:12"}
# entrada = pd.DataFrame(data, index=[0])
# entrada["fecha_vuelo"] = pd.to_datetime(entrada["fecha_vuelo"])
# entrada

In [50]:
# entrada.info()

In [51]:
entrada = pd.DataFrame(data)
entrada["fecha_vuelo"] = pd.to_datetime(entrada["fecha_vuelo"])
entrada.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   aerolinea           16 non-null     object        
 1   aeropuerto_origen   16 non-null     object        
 2   aeropuerto_destino  16 non-null     object        
 3   fecha_vuelo         16 non-null     datetime64[ns]
 4   distancia           8 non-null      float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 772.0+ bytes


In [52]:
r = custom_model.predict(entrada)

In [53]:
# r

In [54]:
for ind, element in enumerate(r):
  if ind % 4 == 0:
    print()
  print("Predicción: ", element['Predicción'], "     Probabilidad de retraso:", round(element['Probabilidad de retraso'],2), "%      Fecha:", entrada.loc[ind]['fecha_vuelo'])


Predicción:  A tiempo      Probabilidad de retraso: 30.26 %      Fecha: 2026-11-30 09:12:00
Predicción:  A tiempo      Probabilidad de retraso: 39.9 %      Fecha: 2026-11-30 13:12:00
Predicción:  A tiempo      Probabilidad de retraso: 37.9 %      Fecha: 2026-11-30 19:12:00
Predicción:  A tiempo      Probabilidad de retraso: 33.2 %      Fecha: 2026-11-30 00:12:00

Predicción:  Retrasado      Probabilidad de retraso: 50.94 %      Fecha: 2026-12-24 09:12:00
Predicción:  Retrasado      Probabilidad de retraso: 53.62 %      Fecha: 2026-12-24 13:12:00
Predicción:  Retrasado      Probabilidad de retraso: 69.17 %      Fecha: 2026-12-24 19:12:00
Predicción:  Retrasado      Probabilidad de retraso: 65.47 %      Fecha: 2026-12-24 00:12:00

Predicción:  Retrasado      Probabilidad de retraso: 52.12 %      Fecha: 2026-01-15 09:12:00
Predicción:  Retrasado      Probabilidad de retraso: 60.27 %      Fecha: 2026-01-15 13:12:00
Predicción:  Retrasado      Probabilidad de retraso: 66.4 %      Fecha: 20

In [55]:
custom_model.explain(entrada.loc[[2]])

,Importancia
hora de vuelo,0.323277
día de la semana,0.234361
aerolinea,0.124225
mes de vuelo,0.103399
aeropuerto_destino,0.057654
aeropuerto_origen,0.045596
distancia_kms,0.023526
fin_de_semana,0.001163
